# Retail Theft Detector - Colab YOLO Training

This notebook downloads a public Kaggle/GitHub-style YOLO dataset, trains a custom Ultralytics YOLO detector on Colab GPU, evaluates it, and saves the trained weights plus metrics report to Google Drive.

Important: this trains the object detector layer. The full theft-review system still needs event-level evaluation later.

## 1. Runtime Setup

In Colab, choose **Runtime > Change runtime type > T4 GPU** before running. You also need a Kaggle API token if using Kaggle:

1. Go to Kaggle > Account > API > Create New Token.
2. Upload `kaggle.json` when the notebook asks for it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, os, shutil, subprocess, textwrap
import numpy as np

# Dataset source options:
# - 'kaggle': needs kaggle.json token
# - 'url_zip': no login if you have a public direct ZIP link
# - 'github_zip': no login for a public GitHub repository ZIP
DATASET_SOURCE = 'kaggle'
KAGGLE_DATASET_SLUG = 'simuletic/cctv-shoplifting-detection-dataset-yolo-and-vlm'
PUBLIC_DATASET_ZIP_URL = ''  # example: https://example.com/dataset.zip
GITHUB_REPO_ZIP_URL = ''      # example: https://github.com/user/repo/archive/refs/heads/main.zip

# Training settings. Start small; increase after the pipeline works.
BASE_MODEL = 'yolo11n.pt'
RUN_NAME = 'retail_theft_yolo11n_v1'
EPOCHS = 50
IMGSZ = 640
BATCH = 16

WORK_DIR = Path('/content/retail_theft_training')
DATA_DIR = WORK_DIR / 'data'
OUTPUT_ROOT = Path('/content/drive/MyDrive/retail_theft_training_outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print('Outputs will be saved to:', OUTPUT_ROOT)

In [ ]:
!pip -q install ultralytics kaggle

import ultralytics
ultralytics.checks()

## 2. Download Dataset

Default dataset: `simuletic/cctv-shoplifting-detection-dataset-yolo-and-vlm`.

If Kaggle blocks access, open the dataset page in your browser, accept its terms/license, then rerun this cell.

In [ ]:
from google.colab import files
from urllib.request import urlretrieve
import zipfile

kaggle_dir = Path('/root/.kaggle')
kaggle_json = kaggle_dir / 'kaggle.json'

if DATASET_SOURCE == 'kaggle':
    if not kaggle_json.exists():
        print('Upload your kaggle.json token now.')
        uploaded = files.upload()
        if 'kaggle.json' not in uploaded:
            raise RuntimeError('kaggle.json was not uploaded.')
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        shutil.move('kaggle.json', kaggle_json)
        os.chmod(kaggle_json, 0o600)

    download_dir = DATA_DIR / 'kaggle_download'
    download_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        'kaggle', 'datasets', 'download',
        '-d', KAGGLE_DATASET_SLUG,
        '-p', str(download_dir),
        '--unzip'
    ], check=True)
elif DATASET_SOURCE in {'url_zip', 'github_zip'}:
    url = PUBLIC_DATASET_ZIP_URL if DATASET_SOURCE == 'url_zip' else GITHUB_REPO_ZIP_URL
    if not url:
        raise ValueError('Set PUBLIC_DATASET_ZIP_URL or GITHUB_REPO_ZIP_URL first.')
    download_dir = DATA_DIR / DATASET_SOURCE
    download_dir.mkdir(parents=True, exist_ok=True)
    zip_path = DATA_DIR / 'dataset.zip'
    print('Downloading:', url)
    urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(download_dir)
else:
    raise ValueError('DATASET_SOURCE must be kaggle, url_zip, or github_zip.')

print('Downloaded files:')
for p in sorted(download_dir.rglob('*'))[:80]:
    print(p.relative_to(download_dir))

## 3. Locate YOLO Dataset Config

The notebook searches for `data.yaml`. If the downloaded dataset has multiple configs, choose the one you want by changing `DATA_YAML`.

In [ ]:
yaml_candidates = sorted(download_dir.rglob('data.yaml')) + sorted(download_dir.rglob('*.yaml'))
yaml_candidates = [p for p in yaml_candidates if p.name.lower() in {'data.yaml', 'dataset.yaml'} or 'data' in p.name.lower()]

if not yaml_candidates:
    raise FileNotFoundError('No YOLO data.yaml found. Inspect the dataset structure and create one manually.')

DATA_YAML = yaml_candidates[0]
print('Using DATA_YAML:', DATA_YAML)
print('\n--- data.yaml preview ---')
print(DATA_YAML.read_text()[:2000])

## 4. Train YOLO

Start with `yolo11n.pt` for speed. If results are weak and Colab GPU memory allows it, try `yolo11s.pt` or `yolo11m.pt`.

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(OUTPUT_ROOT / 'runs' / 'train'),
    name=RUN_NAME,
    device=0,
    patience=15,
    plots=True,
)

TRAIN_SAVE_DIR = Path(train_results.save_dir)
BEST_WEIGHTS = TRAIN_SAVE_DIR / 'weights' / 'best.pt'
LAST_WEIGHTS = TRAIN_SAVE_DIR / 'weights' / 'last.pt'
print('Train save dir:', TRAIN_SAVE_DIR)
print('Best weights:', BEST_WEIGHTS)

## 5. Evaluate on Validation Split

In [ ]:
trained_model = YOLO(str(BEST_WEIGHTS))
metrics = trained_model.val(
    data=str(DATA_YAML),
    split='val',
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(OUTPUT_ROOT / 'runs' / 'val'),
    name=RUN_NAME + '_val',
    device=0,
    plots=True,
)

box = metrics.box
report = {
    'dataset_source': DATASET_SOURCE,
    'kaggle_dataset_slug': KAGGLE_DATASET_SLUG if DATASET_SOURCE == 'kaggle' else '',
    'public_dataset_zip_url': PUBLIC_DATASET_ZIP_URL if DATASET_SOURCE == 'url_zip' else '',
    'github_repo_zip_url': GITHUB_REPO_ZIP_URL if DATASET_SOURCE == 'github_zip' else '',
    'data_yaml': str(DATA_YAML),
    'base_model': BASE_MODEL,
    'best_weights': str(BEST_WEIGHTS),
    'epochs': EPOCHS,
    'imgsz': IMGSZ,
    'batch': BATCH,
    'metrics': {
        'mAP50_95': float(box.map),
        'mAP50': float(box.map50),
        'mAP75': float(box.map75),
        'mean_precision': float(np.mean(box.p)) if len(box.p) else 0.0,
        'mean_recall': float(np.mean(box.r)) if len(box.r) else 0.0,
    },
}

REPORT_PATH = OUTPUT_ROOT / f'{RUN_NAME}_evaluation_report.json'
REPORT_PATH.write_text(json.dumps(report, indent=2), encoding='utf-8')
print(json.dumps(report, indent=2))
print('Saved report:', REPORT_PATH)

## 6. Download/Use Results

The important outputs are saved in Google Drive:

- `best.pt`: trained model weights
- `*_evaluation_report.json`: validation metrics
- training plots/confusion matrices under the run folders

Download `best.pt` and place it in your local project, for example `models/best.pt`. Then configure the local detector to use that path.

In [ ]:
print('Best weights:', BEST_WEIGHTS)
print('Evaluation report:', REPORT_PATH)
print('Drive output folder:', OUTPUT_ROOT)

# Optional direct browser downloads. For large weights, Drive is usually more reliable.
# files.download(str(BEST_WEIGHTS))
# files.download(str(REPORT_PATH))